In [43]:
import torch
import torch.nn as nn
import timm
import pandas as pd
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, confusion_matrix

In [21]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [22]:
dataset = ImageFolder("archive/train", transform=transform)

In [23]:
base_model = timm.create_model('efficientnet_b0', pretrained=True)
features = nn.Sequential(*list(base_model.children())[:-1])
features.eval();

In [28]:
x_list= []
y_list = []

with torch.no_grad():
      for image, label in dataset:
            x = image.unsqueeze(0)
            img_features = features(x)

            x_list.append(img_features)
            y_list.append(label)      

In [33]:
test_dataset = ImageFolder("archive/test", transform=transform)

In [44]:
test_x_list= []
test_y_list = []

with torch.no_grad():
      for image, label in test_dataset:
            x = image.unsqueeze(0)
            img_features = features(x)

            test_x_list.append(img_features)
            test_y_list.append(label)

In [45]:
X_train = torch.cat(x_list)
X_test = torch.cat(test_x_list)

In [49]:
sim = cosine_similarity(X_test, X_train)
sim_like = sim.argmax(axis=1)

pred = []
for i in sim_like:
      pred.append(y_list[i])

print(classification_report(test_y_list, pred))      

              precision    recall  f1-score   support

           0       0.40      0.40      0.40         5
           1       0.75      0.60      0.67         5
           2       0.50      0.60      0.55         5
           3       0.83      1.00      0.91         5
           4       0.50      0.40      0.44         5
           5       0.50      0.40      0.44         5
           6       0.50      0.40      0.44         5
           7       0.33      0.60      0.43         5
           8       0.11      0.20      0.14         5
           9       0.18      0.60      0.27         5
          10       0.25      0.20      0.22         5
          11       0.20      0.20      0.20         5
          12       0.20      0.20      0.20         5
          13       1.00      0.40      0.57         5
          14       0.25      0.20      0.22         5
          15       0.40      0.40      0.40         5
          16       0.43      0.60      0.50         5
          17       0.50    

/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

In [50]:
table = pd.DataFrame(sim.T, index=y_list)
pred2 = table.groupby(level=0).mean().idxmax()

In [52]:
print(classification_report(test_y_list, pred2))

              precision    recall  f1-score   support

           0       0.57      0.80      0.67         5
           1       0.25      0.20      0.22         5
           2       0.33      0.20      0.25         5
           3       0.80      0.80      0.80         5
           4       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         5
           6       0.00      0.00      0.00         5
           7       0.00      0.00      0.00         5
           8       0.25      0.20      0.22         5
           9       0.30      0.60      0.40         5
          10       0.00      0.00      0.00         5
          11       1.00      0.20      0.33         5
          12       0.00      0.00      0.00         5
          13       0.33      0.20      0.25         5
          14       0.00      0.00      0.00         5
          15       0.00      0.00      0.00         5
          16       0.83      1.00      0.91         5
          17       0.50    

/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/oriado/.pyenv/versions/3.13.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

In [53]:
print(confusion_matrix(test_y_list, pred2))

[[4 0 0 ... 0 0 0]
 [0 1 0 ... 1 0 0]
 [0 0 1 ... 0 3 0]
 ...
 [0 0 1 ... 2 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 0 1]]
